# 05 — Cara Kerja Sistem NIDS (peragaan langkah demi langkah)

## Sistem NIDS-nya dalam format apa?

Sistem NIDS **bukan notebook**. Ia adalah **program Python** yang berjalan terus-menerus (24 jam) sebagai
**layanan sistem (systemd)** di server, bersama dashboard web. Notebook tidak cocok untuk itu, karena harus dibuka
dan dijalankan manual dan berhenti saat ditutup.

| Komponen (arsitektur) | Berkas program | Berjalan sebagai |
|---|---|---|
| 1. Penangkap paket | `tcpdump` dipanggil oleh `deteksi.py langsung` | bagian layanan `nids-deteksi` |
| 2. Penyusun aliran & ekstraksi fitur | `nids/pcap.py`, `nids/aliran.py`, `nids/agregasi.py`, `nids/pipa.py` | bagian layanan `nids-deteksi` |
| 3. Modul deteksi Isolation Forest (A + C) | `nids/model.py`, `nids/detektor.py` | bagian layanan `nids-deteksi` |
| 4. Log anomali (SQLite) | `nids/log_anomali.py` → `/var/lib/nids-deteksi/deteksi.db` | bagian layanan `nids-deteksi` |
| 5. Notifikasi Telegram | `nids/telegram.py` | bagian layanan `nids-deteksi` |
| 6. Dashboard web + umpan balik | folder `3_dashboard` (Flask), `nids/umpan_balik.py` | layanan `ta-nids-server` |
| Perekam arsip dataset | folder `1_perekam_traffic` | layanan `traffic-recorder` |
| Pelatihan model | `latih_model.py`, `ubah_ambang.py`, notebook 03/04 | dijalankan manual (laptop / server) |

Lokasi kode di laptop: `D:\KULIAH\TA\04_Kode\2_sistem_nids` (komponen 1–5) dan `D:\KULIAH\TA\04_Kode\3_dashboard`
(komponen 6). Di server: `/opt/nids-deteksi`, `/opt/ta-nids-server`.

**Notebook ini memperagakan alur yang sama** memakai kode yang sama, tetapi paketnya dibaca dari rekaman VPS uji
(bukan langsung dari kartu jaringan), supaya setiap langkah bisa dilihat isinya. Jalankan notebook
`04_Baseline_VPS_Uji.ipynb` lebih dahulu (untuk berkas modelnya).

In [ ]:
import os, sys, glob, time
from datetime import datetime, timezone, timedelta
import pandas as pd

FOLDER = os.path.abspath('')                                     # folder notebook ini (04_Kode/4_notebook)
KODE = os.path.abspath(os.path.join(FOLDER, '..', '2_sistem_nids'))
DATA = os.path.abspath(os.path.join(FOLDER, '..', '..', '05_Data_Eksperimen', 'Server-Uji'))
sys.path.insert(0, KODE)
from nids.pcap import baca_pcap
from nids.pipa import Pipa
from nids.model import ModelDeteksi
from nids.detektor import Detektor
from nids.log_anomali import LogAnomali
from nids.telegram import Notifikasi
from nids.umpan_balik import periksa_ulang, TEKS_STATUS
pd.set_option('display.max_columns', 30)

REKAMAN = os.path.join(DATA, 'rekaman')
MODEL = os.path.join(DATA, 'hasil', 'model_vps_uji.joblib')      # dibuat oleh notebook 04
HASIL = os.path.join(DATA, 'hasil')
os.makedirs(HASIL, exist_ok=True)
WITA = timezone(timedelta(hours=8))
wita = lambda us: datetime.fromtimestamp(us / 1e6, WITA).strftime('%d-%m-%Y %H:%M:%S')
berkas = sorted(glob.glob(os.path.join(REKAMAN, '*.pcap.gz')))
print(len(berkas), 'berkas rekaman VPS uji')

## Langkah 1 — Penangkap paket

Di server, modul deteksi menjalankan `tcpdump -n -p -U -i eth0 -s 160 -w - "tcp or udp"`: hanya 160 byte pertama tiap
paket (header Ethernet + IP + TCP/UDP) yang disalin, lalu dibaca oleh `nids/pcap.py`. Isi komunikasi tidak dibaca;
panjang payload diambil dari header IP. Di sini kita baca satu berkas rekaman (satu jam) yang paling ramai.

In [ ]:
f = max(berkas, key=os.path.getsize)
paket = list(baca_pcap(f))
print('Berkas:', os.path.basename(f), '|', len(paket), 'paket')
pd.DataFrame([{'waktu (WITA)': wita(p.ts), 'IP sumber': p.src, 'port sumber': p.sport, 'IP tujuan': p.dst,
               'port tujuan': p.dport, 'protokol': {6: 'TCP', 17: 'UDP'}.get(p.proto, p.proto),
               'flag TCP': p.flags, 'window': p.win, 'panjang payload': p.payload} for p in paket[:12]])

## Langkah 2 — Penyusun aliran & ekstraksi fitur

Paket dengan IP, port, dan protokol yang sama (dua arah) digabung menjadi satu **aliran**. Saat aliran selesai
(FIN/RST, atau 120 detik), dihitung 68 fitur aliran dan 7 fitur agregasi (perilaku IP sumber dalam 300 detik terakhir).

In [ ]:
pipa = Pipa(300)
aliran, sapu = [], None
for p in paket:
    aliran += pipa.paket(p)
    if sapu is None or p.ts - sapu >= 1_000_000:          # setiap 1 detik: tutup aliran yang kedaluwarsa
        sapu = p.ts
        aliran += pipa.sapu(p.ts)
aliran += pipa.selesai()
fa = pd.DataFrame(aliran)
print('%d paket → %d aliran' % (len(paket), len(fa)))
fa[['Source IP', 'Destination Port', 'Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'agg_ip_aliran', 'agg_ip_port_unik', 'agg_ip_rasio_gagal', 'agg_ip_aliran_ssh']].head(10)

## Langkah 3 — Modul deteksi Isolation Forest (Model A + Model C)

- **Model A** menilai satu aliran dari 68 fitur aliran.
- **Model C** menilai perilaku IP sumber dari 7 fitur agregasi.
- Skor = seberapa "aneh" aliran itu (semakin besar semakin anomali). Aliran ditandai anomali bila
  **skor A > ambang A atau skor C > ambang C**.

In [ ]:
model = ModelDeteksi.muat(MODEL)
print('Model versi %s | ambang A = %.4f | ambang C = %.4f' % (model.versi, model.A.ambang, model.C.ambang))
h = model.nilai(aliran)
fa['skor A'], fa['skor C'], fa['ditandai oleh'] = [x[0] for x in h], [x[1] for x in h], [x[2] or '-' for x in h]
print('Aliran ditandai anomali: %d dari %d' % ((fa['ditandai oleh'] != '-').sum(), len(fa)))
fa.sort_values('skor C', ascending=False)[['Source IP', 'Destination Port', 'agg_ip_aliran', 'agg_ip_port_unik',
                                           'agg_ip_rasio_gagal', 'skor A', 'skor C', 'ditandai oleh']].head(10)

## Langkah 4–5 — Detektor, log anomali (SQLite), dan notifikasi Telegram

Di server, ketiganya digabung oleh `Detektor`: aliran dinilai berkelompok (paling lambat setiap 1 detik), skor semua
aliran dan detail anomali dicatat ke SQLite, lalu notifikasi dikirim (jeda 10 menit per IP sumber).
Di sini kita proses **24 jam terakhir rekaman**, dan pesan Telegram **hanya ditampung, tidak dikirim**.

In [ ]:
DB = os.path.join(HASIL, 'demo_deteksi.db')
for ekor in ('', '-wal', '-shm'):
    if os.path.exists(DB + ekor):
        os.remove(DB + ekor)
log = LogAnomali(DB)
pesan = []
notif = Notifikasi(kirim=lambda teks: (pesan.append(teks), (True, None, None))[1], path_log=DB, jeda_detik=600,
                   nama_server='VPS Uji', jendela_detik=model.jendela_detik, sinkron=True)
det = Detektor(model, log, notif, waktu_log='selesai')
t0 = time.time()
for f in berkas[-24:]:
    for p in baca_pcap(f):
        det.paket(p)
        det.detak(p.ts)
det.selesai()
print('%d paket | %d aliran | %d anomali | %d pesan Telegram | %.1f detik' % (
    det.pipa.jumlah_paket, det.jumlah_aliran, det.jumlah_anomali, len(pesan), time.time() - t0))

In [ ]:
print('Isi tabel di log SQLite (yang dibaca dashboard):')
q = lambda s: pd.read_sql_query(s, log.db)
display(q("SELECT 'aliran' AS tabel, COUNT(*) AS baris FROM aliran UNION ALL SELECT 'anomali', COUNT(*) FROM anomali "
          "UNION ALL SELECT 'ringkasan (per menit)', COUNT(*) FROM ringkasan UNION ALL SELECT 'notifikasi', COUNT(*) FROM notifikasi "
          "UNION ALL SELECT 'model', COUNT(*) FROM model"))
print('IP sumber dengan anomali terbanyak (seperti di dashboard):')
q("SELECT src_ip AS 'IP sumber', COUNT(*) AS anomali, GROUP_CONCAT(DISTINCT oleh) AS model, COUNT(DISTINCT dst_port) AS port, "
  "MAX(agg_ip_port_unik) AS 'port unik maks', MAX(agg_ip_aliran_ssh) AS 'ke SSH maks' FROM anomali GROUP BY src_ip ORDER BY 2 DESC LIMIT 10")

In [ ]:
print('Contoh pesan Telegram yang akan dikirim ke administrator:\n')
print(pesan[0] if pesan else '(tidak ada anomali)')

## Langkah 6 — Dashboard dan loop umpan balik

Dashboard (`ta-nids-server`) membaca log SQLite yang sama dalam mode baca-saja. Bila administrator melaporkan
**serangan lolos**, sistem **memeriksa ulang di modul deteksi**: skor aliran IP itu dinilai ulang dengan ambang yang
diturunkan (paling jauh contamination A 1%, C 5%). Bila tetap tidak terdeteksi → **antrean pelatihan ulang**.
Contoh: kita "laporkan" IP yang menghubungi beberapa port tetapi tidak ditandai.

In [ ]:
calon = q("SELECT src_ip, COUNT(DISTINCT dst_port) AS port, MIN(waktu_selesai_us) AS t0, MAX(waktu_selesai_us) AS t1 "
          "FROM aliran GROUP BY src_ip HAVING SUM(oleh != '') = 0 ORDER BY port DESC LIMIT 1")
if len(calon):
    r = calon.iloc[0]
    hasil = periksa_ulang(log.db, 'lolos', r.src_ip, int(r.t0), int(r.t1), kini_us=int(r.t1) + 60_000_000)
    print('IP dilaporkan :', r.src_ip, '(%d port berbeda, tidak ditandai)' % r.port)
    print('Hasil         :', TEKS_STATUS[hasil['status']])
    display(pd.DataFrame(hasil['kandidat'])[['model', 'skor_maks', 'ambang_sekarang', 'ambang_baru',
                                             'contamination_baru', 'dalam_batas', 'tambahan_aliran_per_hari']])
log.tutup()

## Melihat sistem yang sedang berjalan di VPS uji

```powershell
# dari laptop (PowerShell): masuk ke server + terowongan dashboard
ssh -i $env:USERPROFILE\.ssh\vps_ta_ed25519 -L 8090:127.0.0.1:8090 root@<ip-vps-uji>
```
```bash
systemctl status nids-deteksi           # modul deteksi (langkah 1–5)
journalctl -u nids-deteksi -f           # catatan berjalannya modul
systemctl status ta-nids-server         # dashboard (langkah 6)
sudo bash /opt/traffic-recorder/bin/cek-status.sh   # perekam arsip
```
Lalu buka `http://localhost:8090` di browser laptop → menu **Deteksi** dan **Umpan balik**.